In [ ]:
!pip -q install pdfplumber transformers faiss-cpu google-genai

In [ ]:
import os
import pdfplumber
import numpy as np
import torch
import faiss
import textwrap



In [ ]:
from google.colab import drive

drive.mount('/content/drive')



In [ ]:
pdf_files_path = '/content/drive/MyDrive/DL/RAG_data'

if not os.path.isdir(pdf_files_path):
    raise FileNotFoundError(
        f"PDF folder not found: {pdf_files_path}"
    )

pdf_files = sorted([
    f for f in os.listdir(pdf_files_path)
    if f.lower().endswith('.pdf')
])

if not pdf_files:
    raise ValueError("No PDF files found.")

print("PDF files found:", len(pdf_files))
for filename in pdf_files:
    print(filename)

Extract text and create overlapping chunks

In [ ]:
def extract_text_from_pdf(pdf_path):
    pages = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""

            if text.strip():
                pages.append({
                    "text": text,
                    "source": os.path.basename(pdf_path),
                    "page": page_number
                })

    return pages


def create_chunks(pages, chunk_words=180, overlap=40):
    chunks = []
    step = chunk_words - overlap

    if step <= 0:
        raise ValueError(
            "overlap must be smaller than chunk_words."
        )

    for page in pages:
        words = page["text"].split()

        for start in range(0, len(words), step):
            chunk_text = " ".join(
                words[start:start + chunk_words]
            )

            if chunk_text.strip():
                chunks.append({
                    "text": chunk_text,
                    "source": page["source"],
                    "page": page["page"]
                })

    return chunks


documents = []

for filename in pdf_files:
    pdf_path = os.path.join(pdf_files_path, filename)

    pages = extract_text_from_pdf(pdf_path)
    documents.extend(create_chunks(pages))

if not documents:
    raise ValueError(
        "No text extracted. Scanned PDFs may require OCR."
    )

print("Total chunks:", len(documents))
print("\nFirst chunk:\n", documents[0]["text"][:500])

Load DPR context and question encoders

In [ ]:
from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained(
    "facebook/dpr-ctx_encoder-single-nq-base"
)

ctx_model = DPRContextEncoder.from_pretrained(
    "facebook/dpr-ctx_encoder-single-nq-base"
).to(device)

q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
)

q_model = DPRQuestionEncoder.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
).to(device)

ctx_model.eval()
q_model.eval()

Generate document embeddings

In [ ]:
def encode_documents(documents, batch_size=8):
    all_embeddings = []

    for start in range(0, len(documents), batch_size):
        batch = documents[start:start + batch_size]
        texts = [doc["text"] for doc in batch]

        encoded = ctx_tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.no_grad():
            embeddings = ctx_model(
                **encoded
            ).pooler_output

            embeddings = torch.nn.functional.normalize(
                embeddings, p=2, dim=1
            )

        all_embeddings.append(
            embeddings.cpu().numpy().astype("float32")
        )

    return np.concatenate(all_embeddings, axis=0)


doc_embeddings = encode_documents(documents)

print("Embedding shape:", doc_embeddings.shape)

Create the FAISS similarity index

In [ ]:
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

print("FAISS index created.")
print("Indexed chunks:", index.ntotal)

Retrieve the top-k chunks for a question

In [ ]:
def search(query, k=4):
    encoded_query = q_tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    encoded_query = {
        key: value.to(device)
        for key, value in encoded_query.items()
    }

    with torch.no_grad():
        query_embedding = q_model(
            **encoded_query
        ).pooler_output

        query_embedding = torch.nn.functional.normalize(
            query_embedding, p=2, dim=1
        )

    query_vector = (
        query_embedding.cpu().numpy().astype("float32")
    )

    k = min(k, index.ntotal)
    scores, indices = index.search(query_vector, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        doc = documents[int(idx)]

        results.append({
            "text": doc["text"],
            "source": doc["source"],
            "page": doc["page"],
            "score": float(score)
        })

    return results

In [ ]:
from google import genai
from getpass import getpass

gemini_api_key = getpass(
    "Enter your Gemini API key: "
)

client = genai.Client(api_key=gemini_api_key)

# Change this if the model is unavailable to your account.
MODEL_NAME = "gemini-2.5-flash"

print("Gemini client initialized.")

In [ ]:
from google.genai import types

def generate_rag_answer(query, retrieved_docs):

    if not retrieved_docs:
        return "No relevant document passages were retrieved."

    context_parts = []

    for i, doc in enumerate(retrieved_docs, start=1):
        context_parts.append(
            f"[Source {i}: {doc['source']}, "
            f"page {doc['page']}]\n"
            f"{doc['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a document-grounded question-answering assistant.

Answer the user's question using the supplied document passages.

Instructions:
1. Use the provided context as the primary source.
2. Do not invent facts or unsupported references.
3. If the context is insufficient, state this clearly.
4. Cite source filenames and page numbers where appropriate.
5. For medical questions, provide general information only.
   Do not diagnose the user or replace professional medical advice.

QUESTION:
{query}

RETRIEVED DOCUMENT PASSAGES:
{context}

Write a clear, well-structured answer based on the passages.
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            max_output_tokens=500
        )
    )

    if not response.text:
        return "Gemini did not return a text response."

    return response.text.strip()

In [ ]:
query = (
    "What is lightheadedness and how is it related "
    "to rheumatoid arthritis?"
)

# Step 1: Retrieve relevant chunks
retrieved_docs = search(query, k=4)

print("RETRIEVED DOCUMENT PASSAGES")
print("=" * 70)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nPassage {i}")
    print("Source:", doc["source"])
    print("Page:", doc["page"])
    print("Similarity:", round(doc["score"], 4))
    print(doc["text"][:400])

# Step 2: Generate the answer using Gemini
answer = generate_rag_answer(query, retrieved_docs)

print("\n" + "=" * 70)
print("GEMINI RAG ANSWER")
print("=" * 70)
print(textwrap.fill(answer, width=80))